# ⚡ AI B-Roll Autopilot — Free GPU Backend

Run Whisper transcription + B-roll matching on a **free Colab T4/A100 GPU**, then connect your local (or hosted) web studio to it.

---

### Before you start
1. `Runtime` → `Change runtime type` → select **T4 GPU** → `Save`
2. Add your API keys to **Colab Secrets** (🔑 lock icon in left sidebar):
   - `GEMINI_API_KEY` — [get one here](https://aistudio.google.com/apikey)
   - `PEXELS_API_KEY` — [get one here](https://www.pexels.com/api/) *(optional)*
   - `GIPHY_API_KEY` — [get one here](https://developers.giphy.com/) *(optional)*
3. Run all cells top-to-bottom. Copy the **Public URL** from Cell 3 output and paste it into your web studio settings.

> 💡 **Tip:** The tunnel URL changes every time you restart the notebook.

In [ ]:
# @title 🚀 1. Setup — Clone Repo & Install Dependencies
# ─────────────────────────────────────────────────────────────────────────────
# Verify GPU is available
# ─────────────────────────────────────────────────────────────────────────────
print("🔍 Checking GPU...")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo '⚠️  No GPU detected — switch to T4 GPU runtime!'

# ─────────────────────────────────────────────────────────────────────────────
# Clone the stockpile repo (sorry-i-fucked-up branch)
# ─────────────────────────────────────────────────────────────────────────────
print("\n📦 Cloning repo...")
!rm -rf /content/stockpile
!git clone -q -b sorry-i-fucked-up https://github.com/Sh1sh1R017/stockpile.git /content/stockpile
%cd /content/stockpile
print("✅ Repo cloned.")

In [ ]:
%%capture install_log
# @title    Install system packages & Python dependencies (captured to suppress noise)

import subprocess, sys

# System: ffmpeg
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)

# Python packages
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "pycloudflared",
    "uvicorn[standard]",
    "python-multipart",
    "nest-asyncio",
], check=True)

In [ ]:
# @title    Show install summary
print("✅ All packages installed!")

# Verify key tools
import shutil
for tool in ["ffmpeg", "python3"]:
    path = shutil.which(tool)
    print(f"  {'✅' if path else '❌'} {tool}: {path or 'NOT FOUND'}")

In [ ]:
# @title 🔑 2. Configure API Keys (from Colab Secrets)
# ─────────────────────────────────────────────────────────────────────────────
# RECOMMENDED: Use Colab Secrets (🔑 left sidebar) — keys are stored securely
# and persist across sessions. The fallback below lets you paste them directly
# for quick testing (don't share notebooks with keys pasted in).
# ─────────────────────────────────────────────────────────────────────────────
from pathlib import Path

def _get_secret(name: str, fallback: str = "") -> str:
    """Try Colab Secrets first, then fall back to manual input."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val.strip()
    except Exception:
        pass
    return fallback.strip()

# ── Manual fallbacks (only needed if NOT using Colab Secrets) ──────────────
GEMINI_API_KEY  = ""  # @param {type:"string"}
PEXELS_API_KEY  = ""  # @param {type:"string"}
GIPHY_API_KEY   = ""  # @param {type:"string"}

# ── Whisper model size ────────────────────────────────────────────────────
# small  → fast, lower accuracy  (good for quick tests)
# medium → balanced              (recommended for T4)
# large  → slow, best accuracy   (use on A100)
WHISPER_MODEL        = "medium"  # @param ["small", "medium", "large", "large-v3"]
# float32 is safer on T4 (avoids CUDA precision issues); float16 is faster on A100
WHISPER_COMPUTE_TYPE = "float32"  # @param ["float32", "float16"]

# ── Resolve keys (Secrets → manual fallback) ──────────────────────────────
gemini_key  = _get_secret("GEMINI_API_KEY",  GEMINI_API_KEY)
pexels_key  = _get_secret("PEXELS_API_KEY",  PEXELS_API_KEY)
giphy_key   = _get_secret("GIPHY_API_KEY",   GIPHY_API_KEY)

# ── Write .env ────────────────────────────────────────────────────────────
env_lines = [
    f"LOCAL_INPUT_FOLDER=input",
    f"LOCAL_OUTPUT_FOLDER=output",
    f"WHISPER_MODEL={WHISPER_MODEL}",
    f"WHISPER_COMPUTE_TYPE={WHISPER_COMPUTE_TYPE}",
]
if gemini_key:  env_lines.append(f"GEMINI_API_KEY={gemini_key}")
if pexels_key:  env_lines.append(f"PEXELS_API_KEY={pexels_key}")
if giphy_key:   env_lines.append(f"GIPHY_API_KEY={giphy_key}")

Path("/content/stockpile/.env").write_text("\n".join(env_lines) + "\n", encoding="utf-8")

# ── Summary ───────────────────────────────────────────────────────────────
print("✅ .env written!")
print(f"   GEMINI  : {'✅ set' if gemini_key else '❌ missing — AI director will be disabled'}")
print(f"   PEXELS  : {'✅ set' if pexels_key else '⚠️  not set — B-roll from Pexels disabled'}")
print(f"   GIPHY   : {'✅ set' if giphy_key  else '⚠️  not set — GIF B-roll disabled'}")
print(f"   Whisper : {WHISPER_MODEL} ({WHISPER_COMPUTE_TYPE})")

In [ ]:
# @title 🌐 3. Launch GPU Backend + Public Tunnel
# ─────────────────────────────────────────────────────────────────────────────
# Starts the FastAPI server in a background thread, then creates a
# Cloudflare Quick Tunnel — no account or auth token needed.
# ─────────────────────────────────────────────────────────────────────────────
import os, threading, time, subprocess, nest_asyncio
from pycloudflared import try_cloudflare

nest_asyncio.apply()  # allow asyncio.run() inside Colab's event loop
os.chdir("/content/stockpile")

PORT = 8000

# ── Start FastAPI in background ───────────────────────────────────────────
_server_proc = None

def _start_server():
    global _server_proc
    _server_proc = subprocess.Popen(
        ["python", "-m", "ai_broll_autopilot.cli", "serve",
         "--host", "0.0.0.0", "--port", str(PORT)],
        stdout=open("/tmp/backend.log", "w"),
        stderr=subprocess.STDOUT,
    )

threading.Thread(target=_start_server, daemon=True).start()

# Wait for the server to be ready
import urllib.request
print("⏳ Waiting for backend to start", end="", flush=True)
for _ in range(30):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2)
        print(" ✅", flush=True)
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(2)
else:
    print("\n❌ Backend didn't start in time — check logs:")
    !tail -30 /tmp/backend.log
    raise RuntimeError("Backend failed to start")

# ── Open Cloudflare Quick Tunnel ──────────────────────────────────────────
print("🌐 Opening Cloudflare tunnel...", flush=True)
tunnel = try_cloudflare(port=PORT)
tunnel_url = tunnel.tunnel.url

# ── Print connection info ──────────────────────────────────────────────────
print("\n" + "═" * 65)
print("🚀  AI B-ROLL AUTOPILOT — GPU BACKEND READY")
print("═" * 65)
print(f"  🔗 Public URL  : {tunnel_url}")
print(f"  🖥  Local URL   : http://127.0.0.1:{PORT}")
print(f"  🤖 Whisper     : {WHISPER_MODEL} ({WHISPER_COMPUTE_TYPE})")
print("═" * 65)
print("")
print("📋 TO CONNECT YOUR WEB STUDIO:")
print("  Option A — set env var before starting Next.js dev server:")
print(f'    $env:BACKEND_API_URL="{tunnel_url}"; npm run dev')
print("")
print("  Option B — create/edit  web/.env.local:")
print(f'    BACKEND_API_URL={tunnel_url}')
print("")
print("⚠️  URL changes every restart — update your studio after each new session.")
print("═" * 65)

# ── Keep cell alive & print heartbeat ─────────────────────────────────────
print("\n🟢 Backend is running. This cell will keep it alive.")
print("   Press the ■ Stop button to shut down.\n")
heartbeat = 0
while True:
    time.sleep(60)
    heartbeat += 1
    print(f"   💓 [{heartbeat}m] Alive — {tunnel_url}", flush=True)

In [ ]:
# @title 🛠️ [Optional] Debug — View Backend Logs
# Run this in a separate cell if the backend fails to start
!tail -50 /tmp/backend.log

In [ ]:
# @title 🛠️ [Optional] Health Check
import urllib.request, json
try:
    with urllib.request.urlopen("http://127.0.0.1:8000/health") as r:
        data = json.loads(r.read())
    print("✅ Backend healthy:", data)
except Exception as e:
    print("❌ Backend not reachable:", e)